# 03 - Delta Lake SIRENE : ACID, MERGE, Time Travel, Schema Evolution

**Projet :** Formation Data Engineer, Mois 3 Semaine 2 - PySpark & Delta Lake  
**Stack :** PySpark, Delta Lake, Databricks Free Edition (Azure)  
**Paramètres :** Widgets `env` (dev/prod), `date_partition` (YYYY-MM) - injectés par Airflow

---
| Opération | Détail |
|---|---|
| WRITE initial | 134 661 lignes, 27 colonnes, partition `categorie_entreprise` |
| MERGE (upsert) | 5 lignes mises à jour sur clé `siret`, `tranche_effectif = "99"` |
| Time Travel | `versionAsOf=0`, `timestampAsOf` - 4 versions dans `_delta_log/` |
| Schema evolution | Colonne `source_pipeline` ajoutée via `mergeSchema=True` |
| **Table finale** | **134 666 lignes, 28 colonnes**, `/Volumes/workspace/default/raw_data/sirene_clean_delta` |

In [0]:
# Databricks Widgets
# Injectés par Airflow via notebook_params (DatabricksRunNowOperator)
# Valeurs par défaut utilisées en mode interactif
dbutils.widgets.text("env", "dev", "Environment (dev / prod)")
dbutils.widgets.text("date_partition", "2024-01", "Date partition (YYYY-MM)")

env = dbutils.widgets.get("env")
date_partition = dbutils.widgets.get("date_partition")

print(f"env={env}, date_partition={date_partition}")

env=dev, date_partition=2024-01


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

DELTA_PATH = "/Volumes/workspace/default/raw_data/sirene_clean_delta"
CSV_PATH   = "/Volumes/workspace/default/raw_data/sirene/data.csv"

print(f"Delta path : {DELTA_PATH}")

Delta path : /Volumes/workspace/default/raw_data/sirene_clean_delta


In [0]:
# ── Lecture CSV ───────────────────────────────────────────────────────────────
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(CSV_PATH)
)

# ── Renommage 24 colonnes utiles ──────────────────────────────────────────────
rename_map = {
    "SIREN": "siren",
    "NIC": "nic",
    "SIRET": "siret",
    "Statut de diffusion de l'établissement": "statut_diffusion",
    "Date de création de l'établissement": "date_creation_etab",
    "Tranche de l'effectif de l'établissement": "tranche_effectif",
    "Activité principale de l'établissement8": "activite_principale_etab",
    "Etablissement siège": "etablissement_siege",
    "Code postal de l'établissement": "code_postal",
    "Commune de l'établissement": "commune",
    "Code commune de l'établissement": "code_commune",
    "Code du département de l'établissement": "code_departement",
    "Département de l'établissement": "departement",
    "Code de la région de l'établissement": "code_region",
    "Région de l'établissement": "region",
    "Etat administratif de l'établissement": "etat_admin_etab",
    "Date de fermeture de l'établissement": "date_fermeture_etab",
    "Dénomination de l'unité légale": "denomination_unite_legale",
    "Catégorie de l'entreprise": "categorie_entreprise",
    "Etat administratif de l'unité légale": "etat_admin_ul",
    "Caractère employeur de l'unité légale": "caractere_employeur",
    "Activité principale de l'unité légale": "activite_principale_ul",
    "Catégorie juridique de l'unité légale": "categorie_juridique",
    "Date de création de l'unité légale": "date_creation_ul",
}

df_renamed = df_raw
for old_name, new_name in rename_map.items():
    if old_name in df_renamed.columns:
        df_renamed = df_renamed.withColumnRenamed(old_name, new_name)

df_renamed = df_renamed.select(list(rename_map.values()))

# ── clean_nd ──────────────────────────────────────────────────────────────────
for c in ["denomination_unite_legale", "activite_principale_etab", "categorie_entreprise"]:
    df_renamed = df_renamed.withColumn(
        c,
        F.when(
            (F.col(c) == "[ND]") | (F.col(c) == ""), None
        ).otherwise(F.col(c))
    )

# ── Filtres RGPD ──────────────────────────────────────────────────────────────
df_filtered = (
    df_renamed
    .filter(F.col("etat_admin_etab") == "Actif")
    .filter(F.col("statut_diffusion") != "P")
)

# ── Colonnes dérivées ─────────────────────────────────────────────────────────
df_clean = (
    df_filtered
    .withColumn("loaded_at", F.current_timestamp())
    .withColumn(
        "siret_calcule",
        F.concat(
            F.lpad(F.col("siren").cast("string"), 9, "0"),
            F.lpad(F.col("nic").cast("string"), 5, "0")
        )
    )
    .withColumn(
        "est_siege",
        F.when(F.col("etablissement_siege").isin("oui", "true", "True", "1"), True)
         .when(F.col("etablissement_siege").isin("non", "false", "False", "0"), False)
         .otherwise(None)
    )
)

print(f"df_clean : {df_clean.count():,} lignes, {len(df_clean.columns)} colonnes")

df_clean : 134,661 lignes, 27 colonnes


In [0]:
df_to_write = df_clean.withColumn(
    "categorie_entreprise",
    F.coalesce(F.col("categorie_entreprise"), F.lit("INCONNUE"))
)

# Vérifier la distribution avant écriture - ce seront les dossiers de partition
display(
    df_to_write
    .groupBy("categorie_entreprise")
    .count()
    .orderBy(F.col("count").desc())
)

categorie_entreprise,count
PME,79202
INCONNUE,47024
ETI,5039
GE,3396


In [0]:
(
    df_to_write
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("categorie_entreprise")
    .save(DELTA_PATH)
)

count = spark.read.format("delta").load(DELTA_PATH).count()
print(f"Table Delta écrite : {DELTA_PATH}")
print(f"Lignes : {count:,}, Partition : categorie_entreprise")

Table Delta écrite : /Volumes/workspace/default/raw_data/sirene_clean_delta
Lignes : 134,661, Partition : categorie_entreprise


In [0]:
# Capturer la version juste après l'écriture - sert de référence pour le Time Travel
# (le job tourne en mode "overwrite" à chaque run : la version 0 absolue de la table
#  correspond à sa toute première écriture historique, qui sort vite de la fenêtre
#  de rétention Delta par défaut - deletedFileRetentionDuration = 168h/7 jours.
#  On référence donc la version du run courant, jamais un numéro figé.)
version_avant_merge = (
    DeltaTable.forPath(spark, DELTA_PATH)
    .history(1)
    .collect()[0]["version"]
)
print(f"Version après écriture initiale (ce run) : {version_avant_merge}")

Version après écriture initiale (ce run) : 33


In [0]:
print("Contenu racine de la table Delta")
display(dbutils.fs.ls(DELTA_PATH))
# → Dossiers de partition : categorie_entreprise=GE/
#                            categorie_entreprise=PME/  etc.
# → Dossier _delta_log/  ← le transaction log (ACID)

print("\nTransaction log (_delta_log/)")
display(dbutils.fs.ls(f"{DELTA_PATH}/_delta_log/"))
# → 00000000000000000000.json = version 0 (écriture initiale)
# → Chaque opération future (MERGE, APPEND...) crée un nouveau fichier numéroté

Contenu racine de la table Delta


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/_delta_log/,_delta_log/,0,1785085960626
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/categorie_entreprise=ETI/,categorie_entreprise=ETI/,0,1785085960626
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/categorie_entreprise=GE/,categorie_entreprise=GE/,0,1785085960626
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/categorie_entreprise=INCONNUE/,categorie_entreprise=INCONNUE/,0,1785085960626
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/categorie_entreprise=PME/,categorie_entreprise=PME/,0,1785085960626
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/deletion_vector_0f247068-a516-48b2-bf97-a737add7e4ca.bin,deletion_vector_0f247068-a516-48b2-bf97-a737add7e4ca.bin,79,1783590147000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/deletion_vector_28f2845b-c099-4dda-9f52-026b00bbc6ad.bin,deletion_vector_28f2845b-c099-4dda-9f52-026b00bbc6ad.bin,79,1783589335000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/deletion_vector_2d4dee6b-dc3e-4607-9b21-b2292bab0a94.bin,deletion_vector_2d4dee6b-dc3e-4607-9b21-b2292bab0a94.bin,79,1785084873000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/deletion_vector_4a178b9e-d165-4104-ba8a-98415fae3ecd.bin,deletion_vector_4a178b9e-d165-4104-ba8a-98415fae3ecd.bin,79,1785084756000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/deletion_vector_65f7a608-6dac-4177-9975-5fb06820b1ec.bin,deletion_vector_65f7a608-6dac-4177-9975-5fb06820b1ec.bin,79,1785085277000



Transaction log (_delta_log/)


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/_delta_log/00000000000000000000.crc,00000000000000000000.crc,14369,1783438406000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/_delta_log/00000000000000000000.json,00000000000000000000.json,13503,1783438406000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/_delta_log/00000000000000000001.crc,00000000000000000001.crc,14369,1783438436000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/_delta_log/00000000000000000001.json,00000000000000000001.json,12414,1783438436000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/_delta_log/00000000000000000002.crc,00000000000000000002.crc,14369,1783439688000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/_delta_log/00000000000000000002.json,00000000000000000002.json,4417,1783439687000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/_delta_log/00000000000000000003.crc,00000000000000000003.crc,4564,1783439983000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/_delta_log/00000000000000000003.json,00000000000000000003.json,5918,1783439983000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/_delta_log/00000000000000000004.crc,00000000000000000004.crc,4476,1783589315000
dbfs:/Volumes/workspace/default/raw_data/sirene_clean_delta/_delta_log/00000000000000000004.json,00000000000000000004.json,15466,1783589314000


In [0]:
# Lecture DataFrame
df_delta = spark.read.format("delta").load(DELTA_PATH)
print(f"Lignes Delta : {df_delta.count():,}")
print(f"Colonnes : {len(df_delta.columns)}")
display(df_delta.limit(5))

# Spark SQL - syntaxe backtick-path (native Delta)
df_stats = spark.sql(f"""
    SELECT
        categorie_entreprise,
        COUNT(*)                                            AS nb_etabs,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct_total
    FROM delta.`{DELTA_PATH}`
    GROUP BY categorie_entreprise
    ORDER BY nb_etabs DESC
""")
display(df_stats)

Lignes Delta : 134,661
Colonnes : 27


siren,nic,siret,statut_diffusion,date_creation_etab,tranche_effectif,activite_principale_etab,etablissement_siege,code_postal,commune,code_commune,code_departement,departement,code_region,region,etat_admin_etab,date_fermeture_etab,denomination_unite_legale,categorie_entreprise,etat_admin_ul,caractere_employeur,activite_principale_ul,categorie_juridique,date_creation_ul,loaded_at,siret_calcule,est_siege
808719801,33,80871980100033,O,2020-05-04,Etablissement non employeur,null,oui,44200,NANTES,44109,44,Loire-Atlantique,52,Pays de la Loire,Actif,null,null,PME,Active,null,69.10Z,1000,2015-01-05,2026-07-26T17:12:35.491Z,80871980100033,true
892492018,15,89249201800015,O,2021-01-02,Etablissement non employeur,null,oui,44470,THOUARE-SUR-LOIRE,44204,44,Loire-Atlantique,52,Pays de la Loire,Actif,null,null,PME,Active,null,47.91A,1000,2021-01-02,2026-07-26T17:12:35.491Z,89249201800015,true
501700744,37,50170074400037,O,2019-10-03,3 à 5 salariés,null,oui,44000,NANTES,44109,44,Loire-Atlantique,52,Pays de la Loire,Actif,null,SELARL PUBLI - JURIS,PME,Active,null,69.10Z,5485,2007-01-01,2026-07-26T17:12:35.491Z,50170074400037,true
909580870,14,90958087000014,O,2022-01-18,Etablissement non employeur,null,oui,44100,NANTES,44109,44,Loire-Atlantique,52,Pays de la Loire,Actif,null,SCI 31 NICOLAS APPERT,PME,Active,null,68.20B,6540,2022-01-18,2026-07-26T17:12:35.491Z,90958087000014,true
919498055,12,91949805500012,O,2022-10-01,Etablissement non employeur,null,oui,44640,SAINT-JEAN-DE-BOISEAU,44166,44,Loire-Atlantique,52,Pays de la Loire,Actif,null,null,PME,Active,null,43.33Z,1000,2022-10-01,2026-07-26T17:12:35.491Z,91949805500012,true


categorie_entreprise,nb_etabs,pct_total
PME,79202,58.8
INCONNUE,47024,34.9
ETI,5039,3.7
GE,3396,2.5


In [0]:
# Lire 5 lignes existantes depuis Delta et simuler une mise à jour
df_updates = (
    spark.read.format("delta").load(DELTA_PATH)
    .limit(5)
    .withColumn("tranche_effectif", F.lit("99"))   # simulation : 5 000+ salariés
    .withColumn("loaded_at", F.current_timestamp())
)

# Garder les SIRETs modifiés en mémoire - utilisés dans les étapes 9, 11, 12
sirets_modifies = [row.siret for row in df_updates.select("siret").collect()]

print(f"SIRETs à mettre à jour ({len(sirets_modifies)}) :")
for s in sirets_modifies:
    print(f"{s}")

display(df_updates.select("siret", "denomination_unite_legale", "tranche_effectif"))

SIRETs à mettre à jour (5) :
80871980100033
89249201800015
50170074400037
90958087000014
91949805500012


siret,denomination_unite_legale,tranche_effectif
80871980100033,null,99
89249201800015,null,99
50170074400037,SELARL PUBLI - JURIS,99
90958087000014,SCI 31 NICOLAS APPERT,99
91949805500012,null,99


In [0]:
delta_table = DeltaTable.forPath(spark, DELTA_PATH)

(
    delta_table.alias("target")
    .merge(
        df_updates.alias("source"),
        "target.siret = source.siret"
    )
    .whenMatchedUpdateAll()    # met à jour TOUTES les colonnes si siret matché
    .execute()
)

print("MERGE exécuté - 5 lignes mises à jour (tranche_effectif → '99')")

MERGE exécuté - 5 lignes mises à jour (tranche_effectif → '99')


In [0]:
df_verification = (
    spark.read.format("delta").load(DELTA_PATH)
    .filter(F.col("siret").isin(sirets_modifies))
    .select("siret", "denomination_unite_legale", "tranche_effectif", "loaded_at")
)
display(df_verification)
# tranche_effectif = "99" pour les 5 lignes
# loaded_at = timestamp du MERGE (plus récent que les autres lignes)

siret,denomination_unite_legale,tranche_effectif,loaded_at
80871980100033,null,99,2026-07-26T17:12:47.347Z
89249201800015,null,99,2026-07-26T17:12:47.347Z
50170074400037,SELARL PUBLI - JURIS,99,2026-07-26T17:12:47.347Z
90958087000014,SCI 31 NICOLAS APPERT,99,2026-07-26T17:12:47.347Z
91949805500012,null,99,2026-07-26T17:12:47.347Z


In [0]:
delta_table = DeltaTable.forPath(spark, DELTA_PATH)

display(delta_table.history())
# Colonnes clés à observer :
# → version    : le numéro croît à chaque run (overwrite/MERGE/append s'accumulent
#                dans le même _delta_log/, jamais remis à 0)
# → timestamp  : horodatage de chaque opération
# → operation  : "WRITE" / "MERGE"
# → operationMetrics : nb de lignes insérées/mises à jour

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
34,2026-07-26T17:12:51.000Z,74238550927607,brucher.alan@gmail.com,MERGE,"Map(predicate -> [""(siret#29775L = siret#29685L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(570634775339913),0b0cfa95-01ce-4ec6-9d34-484bffb257f8,0726-165135-k6sz0g1o-v2n,33,WriteSerializable,false,"Map(numTargetRowsCopied -> 79197, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3181129, numTargetBytesRemoved -> 3181045, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3260, materializeSourceTimeMs -> 181, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1165, numTargetRowsUpdated -> 5, numOutputRows -> 79202, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1824)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
33,2026-07-26T17:12:38.000Z,74238550927607,brucher.alan@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [""categorie_entreprise""], canOverwriteSchema -> true)",null,List(570634775339913),397b9316-d868-414d-9de6-8551bfeef13b,0726-165135-k6sz0g1o-v2n,32,WriteSerializable,false,"Map(numFiles -> 4, numRemovedFiles -> 5, numRemovedBytes -> 5661586, numDeletionVectorsRemoved -> 0, numOutputRows -> 134661, numOutputBytes -> 5653442)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
32,2026-07-26T17:06:21.000Z,74238550927607,brucher.alan@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [], canMergeSchema -> true)","List(453432341513798, sirene_spark_transform, 983352337493630, 446978846268832, 74238550927607, manual)",List(570634775339913),a59d5197-3792-4c54-9848-cc93d7d547e4,0726-165135-k6sz0g1o-v2n,31,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 7852)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
31,2026-07-26T17:06:10.000Z,74238550927607,brucher.alan@gmail.com,MERGE,"Map(predicate -> [""(siret#25840L = siret#25750L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])","List(453432341513798, sirene_spark_transform, 983352337493630, 446978846268832, 74238550927607, manual)",List(570634775339913),ddc36b6a-c59d-45c9-a56c-aebcbe02970e,0726-165135-k6sz0g1o-v2n,30,WriteSerializable,false,"Map(numTargetRowsCopied -> 79197, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3181181, numTargetBytesRemoved -> 3181097, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3173, materializeSourceTimeMs -> 186, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1089, numTargetRowsUpdated -> 5, numOutputRows -> 79202, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1808)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
30,2026-07-26T17:05:58.000Z,74238550927607,brucher.alan@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [""categorie_entreprise""], canOverwriteSchema -> true)","List(453432341513798, sirene_spark_transform, 983352337493630, 446978846268832, 74238550927607, manual)",List(570634775339913),51d7f71c-420f-48cf-a0f5-a21504009906,0726-165135-k6sz0g1o-v2n,29,WriteSerializable,false,"Map(numFiles -

In [0]:
# Lire la version capturée juste après l'écriture initiale (ce run) - état avant MERGE
df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", version_avant_merge)
    .load(DELTA_PATH)
)

# Lire la version courante - état après MERGE
df_v1 = spark.read.format("delta").load(DELTA_PATH)

print(f"Version {version_avant_merge} - avant MERGE")
display(
    df_v0.filter(F.col("siret").isin(sirets_modifies))
         .select("siret", "denomination_unite_legale", "tranche_effectif")
)

print("Version courante - après MERGE")
display(
    df_v1.filter(F.col("siret").isin(sirets_modifies))
         .select("siret", "denomination_unite_legale", "tranche_effectif")
)

Version 33 - avant MERGE


siret,denomination_unite_legale,tranche_effectif
80871980100033,null,Etablissement non employeur
89249201800015,null,Etablissement non employeur
50170074400037,SELARL PUBLI - JURIS,3 à 5 salariés
90958087000014,SCI 31 NICOLAS APPERT,Etablissement non employeur
91949805500012,null,Etablissement non employeur


Version courante - après MERGE


siret,denomination_unite_legale,tranche_effectif
80871980100033,null,99
89249201800015,null,99
50170074400037,SELARL PUBLI - JURIS,99
90958087000014,SCI 31 NICOLAS APPERT,99
91949805500012,null,99


In [0]:
# Récupérer le timestamp de la version capturée (ce run) depuis l'historique
ts_v0 = (
    DeltaTable.forPath(spark, DELTA_PATH)
    .history()
    .filter(F.col("version") == version_avant_merge)
    .collect()[0]["timestamp"]
)
print(f"Timestamp version {version_avant_merge} : {ts_v0}")

# Lire par timestamp - même résultat que versionAsOf=version_avant_merge
df_by_ts = (
    spark.read
    .format("delta")
    .option("timestampAsOf", str(ts_v0))
    .load(DELTA_PATH)
)
print(f"Lignes version {version_avant_merge} par timestamp : {df_by_ts.count():,}")
# 134 661 (identique à versionAsOf=version_avant_merge)

Timestamp version 33 : 2026-07-26 17:12:38
Lignes version 33 par timestamp : 134,661


In [0]:
# 5 nouvelles lignes avec la colonne source_pipeline supplémentaire
df_new_rows = (
    spark.read.format("delta").load(DELTA_PATH)
    .limit(5)
    .withColumn("source_pipeline", F.lit(f"databricks_m3_{env}_{date_partition}"))
    .withColumn("loaded_at", F.current_timestamp())
)

(
    df_new_rows
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")   # ← autorise l'ajout de colonne
    .save(DELTA_PATH)
)

print("Schema evolution - colonne 'source_pipeline' ajoutée")

Schema evolution - colonne 'source_pipeline' ajoutée


In [0]:
df_final = spark.read.format("delta").load(DELTA_PATH)

print(f"Total lignes : {df_final.count():,}")             # → 134 666 (134 661 + 5)
print(f"Colonnes : {len(df_final.columns)}")            # → 28 (source_pipeline ajoutée)

# Distribution source_pipeline
display(df_final.groupBy("source_pipeline").count())
# → null                : 134 661 lignes (les originales)
# → databricks_free_m3  :       5 lignes (schema evolution)

# Historique final de ce run - 3 nouvelles opérations
display(
    DeltaTable.forPath(spark, DELTA_PATH)
    .history()
    .select("version", "timestamp", "operation")
)
# → version_avant_merge     : WRITE  (écriture initiale, ce run)
# → version_avant_merge + 1 : MERGE  (upsert)
# → version_avant_merge + 2 : WRITE  (append schema evolution)

Total lignes : 134,666
Colonnes : 28


source_pipeline,count
null,134661
databricks_m3_dev_2024-01,5


version,timestamp,operation
35,2026-07-26T17:13:02.000Z,WRITE
34,2026-07-26T17:12:51.000Z,MERGE
33,2026-07-26T17:12:38.000Z,WRITE
32,2026-07-26T17:06:21.000Z,WRITE
31,2026-07-26T17:06:10.000Z,MERGE
30,2026-07-26T17:05:58.000Z,WRITE
29,2026-07-26T17:04:32.000Z,WRITE
28,2026-07-26T17:04:23.000Z,MERGE
27,2026-07-26T17:04:10.000Z,WRITE
26,2026-07-26T17:01:19.000Z,MERGE
